# Week 5 Silver Transformation (Spark SQL)

This notebook creates trusted Silver Delta tables from the Bronze tables created in the previous notebook.

### Silver-layer work performed
- Standardize column values using `TRIM`, `UPPER`, and consistent data types
- Convert date/timestamp fields to proper Spark SQL types
- Remove invalid/blank business keys
- Deduplicate records using business keys and the latest available record
- Flatten the nested customer `accounts` array into a separate `silver_customer_accounts` table
- Keep useful lineage columns from Bronze


## 1. Customers — bronze_customers → silver_customers

In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_customers
USING DELTA AS
WITH ranked AS (
    SELECT
        customer_id,
        customer_record_id,
        TRIM(customer_segment) AS customer_segment,
        UPPER(TRIM(customer_status)) AS customer_status,
        UPPER(TRIM(home_country_code)) AS home_country_code,
        UPPER(TRIM(risk_profile)) AS risk_profile,
        CAST(created_timestamp AS TIMESTAMP) AS created_timestamp,
        source_file_name,
        ingestion_timestamp,
        ingestion_run_id,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY ingestion_timestamp DESC
        ) AS rn
    FROM data_engineering.default.bronze_customers
    WHERE customer_id IS NOT NULL
      AND TRIM(customer_id) <> ''
)
SELECT
    customer_id,
    customer_record_id,
    customer_segment,
    customer_status,
    home_country_code,
    risk_profile,
    created_timestamp,
    source_file_name,
    ingestion_timestamp,
    ingestion_run_id
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_customer_accounts
USING DELTA AS
WITH customer_accounts AS (
    SELECT
        customer_id,
        EXPLODE(accounts) AS account
    FROM data_engineering.default.bronze_customers
    WHERE customer_id IS NOT NULL
      AND accounts IS NOT NULL
),
ranked AS (
    SELECT
        account.account_id AS account_id,
        account.account_record_id AS account_record_id,
        customer_id,
        UPPER(TRIM(account.account_status)) AS account_status,
        UPPER(TRIM(account.account_type)) AS account_type,
        CAST(account.daily_limit_reporting AS DECIMAL(18,2)) AS daily_limit_reporting,
        CAST(account.opened_date AS DATE) AS opened_date,
        UPPER(TRIM(account.reporting_currency)) AS reporting_currency,
        ROW_NUMBER() OVER (
            PARTITION BY account.account_id
            ORDER BY customer_id
        ) AS rn
    FROM customer_accounts
    WHERE account.account_id IS NOT NULL
      AND TRIM(account.account_id) <> ''
)
SELECT
    account_id,
    account_record_id,
    customer_id,
    account_status,
    account_type,
    daily_limit_reporting,
    opened_date,
    reporting_currency
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_customers_count
FROM data_engineering.default.silver_customers;


silver_customers_count
18000


In [0]:
%sql
SELECT COUNT(*) AS silver_customer_accounts_count
FROM data_engineering.default.silver_customer_accounts;


silver_customer_accounts_count
22000


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_customers
LIMIT 10;


customer_id,customer_record_id,customer_segment,customer_status,home_country_code,risk_profile,created_timestamp,source_file_name,ingestion_timestamp,ingestion_run_id
CUS0000001,CUSR00000001,FAMILY_BUDGET,ACTIVE,GB,LOW,2022-01-10T01:18:39.000Z,customers.json,2026-07-31T10:04:02.033Z,a945fe42-c7c7-48f6-bfbe-8c87f433b04e
CUS0000002,CUSR00000002,DIGITAL_REGULAR,ACTIVE,IN,LOW,2023-08-15T03:31:47.000Z,customers.json,2026-07-31T10:04:02.033Z,a9857bc2-afb3-4bec-8345-9ef013f2fe06
CUS0000003,CUSR00000003,PREMIUM,ACTIVE,IN,LOW,2023-04-11T17:23:47.000Z,customers.json,2026-07-31T10:04:02.033Z,8adcac10-4919-483b-a04d-ad7dfac68f16
CUS0000004,CUSR00000004,DIGITAL_REGULAR,ACTIVE,US,MEDIUM,2023-12-25T10:12:52.000Z,customers.json,2026-07-31T10:04:02.033Z,6a751987-046c-4c91-8074-ba9f8a82bc90
CUS0000005,CUSR00000005,SMALL_BUSINESS,ACTIVE,IN,MEDIUM,2022-11-13T07:39:00.000Z,customers.json,2026-07-31T10:04:02.033Z,4272f9d0-da8d-4e83-9604-7a314ce5c366
CUS0000006,CUSR00000006,FAMILY_BUDGET,ACTIVE,IN,LOW,2023-05-24T13:00:22.000Z,customers.json,2026-07-31T10:04:02.033Z,079df9b4-04fa-4e7f-8896-c798fab68c1a
CUS0000007,CUSR00000007,SMALL_BUSINESS,ACTIVE,IN,MEDIUM,2023-07-30T18:58:31.000Z,customers.json,2026-07-31T10:04:02.033Z,af7fcc21-1103-4510-a8a0-b82ef275ab83
CUS0000008,CUSR00000008,TRAVEL_ACTIVE,ACTIVE,IN,LOW,2022-06-05T06:23:40.000Z,customers.json,2026-07-31T10:04:02.033Z,b91ee87d-3561-4d6d-89e7-f3812afdd44a
CUS0000009,CUSR00000009,SMALL_BUSINESS,SUSPENDED,IN,LOW,2023-02-09T20:42:56.000Z,customers.json,2026-07-31T10:04:02.033Z,e7850eaf-d64a-4362-8154-4551f3fcdf64
CUS0000010,CUSR00000010,SMALL_BUSINESS,ACTIVE,US,MEDIUM,2025-09-15T05:29:57.000Z,customers.json,2026-07-31T10:04:02.033Z,e652d3d0-25ca-446b-98d5-17a940bb6697


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_customer_accounts
LIMIT 10;


account_id,account_record_id,customer_id,account_status,account_type,daily_limit_reporting,opened_date,reporting_currency
ACC00000001,ACCR00000001,CUS0005582,ACTIVE,SAVINGS,53875.98,2025-06-17,USD
ACC00000002,ACCR00000002,CUS0009481,ACTIVE,SAVINGS,39629.94,2023-07-16,INR
ACC00000003,ACCR00000003,CUS0005670,ACTIVE,CURRENT,16686.69,2024-06-21,GBP
ACC00000004,ACCR00000004,CUS0003342,ACTIVE,CURRENT,76319.02,2022-04-21,INR
ACC00000005,ACCR00000005,CUS0000223,ACTIVE,CURRENT,194132.29,2022-05-03,INR
ACC00000006,ACCR00000006,CUS0003782,ACTIVE,SAVINGS,84597.19,2024-09-16,INR
ACC00000007,ACCR00000007,CUS0002724,ACTIVE,SAVINGS,77179.65,2022-03-15,INR
ACC00000008,ACCR00000008,CUS0002095,CLOSED,SAVINGS,290484.77,2025-07-27,INR
ACC00000009,ACCR00000009,CUS0010144,ACTIVE,CURRENT,72484.32,2023-06-23,INR
ACC00000010,ACCR00000010,CUS0005456,ACTIVE,CURRENT,68371.38,2022-01-02,SGD


## 2. Devices — bronze_devices → silver_devices

In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_devices
USING DELTA AS
WITH ranked AS (
    SELECT
        TRIM(device_record_id) AS device_record_id,
        TRIM(device_id) AS device_id,
        TRIM(customer_id) AS customer_id,
        TRIM(device_type) AS device_type,
        TRIM(os_family) AS os_family,
        UPPER(TRIM(device_trust_status)) AS device_trust_status,
        CAST(first_seen_timestamp AS TIMESTAMP) AS first_seen_timestamp,
        CAST(last_seen_timestamp AS TIMESTAMP) AS last_seen_timestamp,
        TRIM(device_fingerprint_token) AS device_fingerprint_token,
        source_file_name,
        ingestion_timestamp,
        ingestion_run_id,
        ROW_NUMBER() OVER (
            PARTITION BY device_id
            ORDER BY last_seen_timestamp DESC, ingestion_timestamp DESC
        ) AS rn
    FROM data_engineering.default.bronze_devices
    WHERE device_id IS NOT NULL
      AND TRIM(device_id) <> ''
)
SELECT
    device_record_id,
    device_id,
    customer_id,
    device_type,
    os_family,
    device_trust_status,
    first_seen_timestamp,
    last_seen_timestamp,
    device_fingerprint_token,
    source_file_name,
    ingestion_timestamp,
    ingestion_run_id
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_devices_count
FROM data_engineering.default.silver_devices;


silver_devices_count
30000


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_devices
LIMIT 10;


device_record_id,device_id,customer_id,device_type,os_family,device_trust_status,first_seen_timestamp,last_seen_timestamp,device_fingerprint_token,source_file_name,ingestion_timestamp,ingestion_run_id
DEVR00000001,DEV00000001,CUS0016346,MOBILE,WINDOWS,TRUSTED,2024-12-27T19:33:05.000Z,2025-04-19T05:16:10.000Z,TOK-D-0000000001,devices.csv,2026-07-31T10:04:47.073Z,02215ba4-7891-48f7-8428-9c4e16494a50
DEVR00000002,DEV00000002,CUS0016513,MOBILE,EMBEDDED,TRUSTED,2024-07-17T13:10:42.000Z,2025-04-04T14:31:56.000Z,TOK-D-0000000002,devices.csv,2026-07-31T10:04:47.073Z,6afe783d-4a27-42fb-89ef-aac8afc1a70f
DEVR00000003,DEV00000003,CUS0013927,DESKTOP,EMBEDDED,TRUSTED,2025-03-19T19:48:47.000Z,2025-09-12T00:50:51.000Z,TOK-D-0000000003,devices.csv,2026-07-31T10:04:47.073Z,b4831550-eeb6-45c9-bd92-3f46c514e0c2
DEVR00000004,DEV00000004,CUS0010054,MOBILE,WINDOWS,TRUSTED,2025-01-03T18:53:00.000Z,2025-08-09T16:56:04.000Z,TOK-D-0000000004,devices.csv,2026-07-31T10:04:47.073Z,a077ad2a-5cfa-477b-a8fc-e10721f95f3e
DEVR00000005,DEV00000005,CUS0001336,MOBILE,IOS,TRUSTED,2026-04-26T15:10:44.000Z,2026-06-30T23:59:59.000Z,TOK-D-0000000005,devices.csv,2026-07-31T10:04:47.073Z,621afa9c-54eb-4693-a3c4-a2ac432b79b9
DEVR00000006,DEV00000006,CUS0003675,DESKTOP,LINUX,TRUSTED,2025-02-18T22:18:47.000Z,2025-05-03T11:21:00.000Z,TOK-D-0000000006,devices.csv,2026-07-31T10:04:47.073Z,6c987e49-0d11-4c87-94bc-1f6e7f46992f
DEVR00000007,DEV00000007,CUS0013039,DESKTOP,LINUX,TRUSTED,2025-07-16T10:46:39.000Z,2025-09-10T03:48:35.000Z,TOK-D-0000000007,devices.csv,2026-07-31T10:04:47.073Z,e2f1cc8c-c9d0-4a57-b47d-1e2e03dd4a14
DEVR00000008,DEV00000008,CUS0013238,MOBILE,LINUX,TRUSTED,2024-01-31T07:44:04.000Z,2024-11-23T02:16:09.000Z,TOK-D-0000000008,devices.csv,2026-07-31T10:04:47.073Z,98d85622-5097-4fa3-b5c0-30beeb45256a
DEVR00000009,DEV00000009,CUS0007242,TABLET,IOS,TRUSTED,2024-01-13T05:48:04.000Z,2024-03-26T06:08:29.000Z,TOK-D-0000000009,devices.csv,2026-07-31T10:04:47.073Z,b4cdef2e-0c1b-4e56-a4c5-baecbb069237
DEVR00000010,DEV00000010,CUS0017718,MOBILE,LINUX,TRUSTED,2025-01-25T09:32:56.000Z,2025-04-25T22:21:46.000Z,TOK-D-0000000010,devices.csv,2026-07-31T10:04:47.073Z,95443db0-4a6e-441a-93bb-cb30edde9a0c


## 3. Fraud Cases — bronze_fraud_cases → silver_fraud_cases

In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_fraud_cases
USING DELTA AS
WITH ranked AS (
    SELECT
        TRIM(case_record_id) AS case_record_id,
        TRIM(case_id) AS case_id,
        TRIM(primary_transaction_id) AS primary_transaction_id,
        TRIM(related_transaction_ids) AS related_transaction_ids,
        CAST(case_created_timestamp AS TIMESTAMP) AS case_created_timestamp,
        CAST(review_started_timestamp AS TIMESTAMP) AS review_started_timestamp,
        CAST(case_closed_timestamp AS TIMESTAMP) AS case_closed_timestamp,
        UPPER(TRIM(case_status)) AS case_status,
        UPPER(TRIM(case_priority)) AS case_priority,
        UPPER(TRIM(final_disposition)) AS final_disposition,
        TRIM(review_queue) AS review_queue,
        TRIM(rule_trigger_summary) AS rule_trigger_summary,
        source_file_name,
        ingestion_timestamp,
        ingestion_run_id,
        ROW_NUMBER() OVER (
            PARTITION BY case_id
            ORDER BY case_closed_timestamp DESC NULLS LAST,
                     ingestion_timestamp DESC
        ) AS rn
    FROM data_engineering.default.bronze_fraud_cases
    WHERE case_id IS NOT NULL
      AND TRIM(case_id) <> ''
)
SELECT
    case_record_id,
    case_id,
    primary_transaction_id,
    related_transaction_ids,
    case_created_timestamp,
    review_started_timestamp,
    case_closed_timestamp,
    case_status,
    case_priority,
    final_disposition,
    review_queue,
    rule_trigger_summary,
    source_file_name,
    ingestion_timestamp,
    ingestion_run_id
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_fraud_cases_count
FROM data_engineering.default.silver_fraud_cases;


silver_fraud_cases_count
4200


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_fraud_cases
LIMIT 10;


case_record_id,case_id,primary_transaction_id,related_transaction_ids,case_created_timestamp,review_started_timestamp,case_closed_timestamp,case_status,case_priority,final_disposition,review_queue,rule_trigger_summary,source_file_name,ingestion_timestamp,ingestion_run_id
CASR00000001,CAS0000001,TXN0000095676,TXN0000095676,2026-03-01T01:06:08.000Z,2026-03-01T05:40:57.000Z,2026-03-03T15:42:39.000Z,CLOSED_LEGITIMATE,MEDIUM,LEGITIMATE,QUEUE_B,SYN-R03_CROSS_BORDER|SYN-R04_MERCHANT_RISK,fraud_cases.csv,2026-07-31T10:05:13.306Z,97525aa8-c5a8-4c0c-9305-32f117e66561
CASR00000002,CAS0000002,TXN0000197782,TXN0000197782,2026-05-20T21:06:36.000Z,2026-05-20T21:32:51.000Z,2026-05-22T07:22:08.000Z,CLOSED_LEGITIMATE,MEDIUM,LEGITIMATE,QUEUE_D,SYN-R03_CROSS_BORDER|SYN-R04_MERCHANT_RISK|SYN-R06_LOCATION_CHANGE,fraud_cases.csv,2026-07-31T10:05:13.306Z,6aa72468-9522-4cb4-9762-6ffac3fabcbf
CASR00000003,CAS0000003,TXN0000210539,TXN0000210539,2026-04-06T07:13:11.000Z,2026-04-06T09:27:22.000Z,2026-04-11T03:46:33.000Z,CLOSED_CONFIRMED,MEDIUM,CONFIRMED_FRAUD,QUEUE_A,SYN-R01_NEW_DEVICE|SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,fraud_cases.csv,2026-07-31T10:05:13.306Z,2307ea0c-dbfa-46e3-aad0-c24b36a7dced
CASR00000004,CAS0000004,TXN0000043911,TXN0000043911,2026-04-07T01:17:43.000Z,2026-04-07T06:09:40.000Z,2026-04-11T21:29:07.000Z,CLOSED_LEGITIMATE,MEDIUM,LEGITIMATE,QUEUE_A,SYN-R03_CROSS_BORDER,fraud_cases.csv,2026-07-31T10:05:13.306Z,850e047a-d889-4aa9-80ae-91508bf7a1c9
CASR00000005,CAS0000005,TXN0000007642,TXN0000007642,2026-05-20T20:28:41.000Z,2026-05-21T00:04:34.000Z,2026-05-25T04:13:43.000Z,CLOSED_CONFIRMED,MEDIUM,CONFIRMED_FRAUD,QUEUE_C,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,fraud_cases.csv,2026-07-31T10:05:13.306Z,d4a65a10-9d96-42fb-997c-d89b3643e9cc
CASR00000006,CAS0000006,TXN0000091267,TXN0000091267,2026-06-09T00:32:17.000Z,2026-06-09T04:45:22.000Z,null,OPEN,MEDIUM,PENDING,QUEUE_B,SYN-R00_BASELINE,fraud_cases.csv,2026-07-31T10:05:13.306Z,5787c6f6-6205-46da-b296-02c3f349890a
CASR00000007,CAS0000007,TXN0000016313,TXN0000016313,2026-02-22T10:44:53.000Z,2026-02-22T12:00:13.000Z,2026-02-23T19:00:56.000Z,CLOSED_INCONCLUSIVE,MEDIUM,INCONCLUSIVE,QUEUE_D,SYN-R01_NEW_DEVICE|SYN-R03_CROSS_BORDER,fraud_cases.csv,2026-07-31T10:05:13.306Z,e19583f5-c4fe-44f9-a0ed-3838823c9ae7
CASR00000008,CAS0000008,TXN0000082784,TXN0000082784,2026-01-14T17:38:49.000Z,2026-01-14T20:50:23.000Z,2026-01-18T08:02:37.000Z,CLOSED_INCONCLUSIVE,MEDIUM,INCONCLUSIVE,QUEUE_D,SYN-R01_NEW_DEVICE|SYN-R03_CROSS_BORDER,fraud_cases.csv,2026-07-31T10:05:13.306Z,f8954bae-3683-4592-b570-f979a8f5fa5a
CASR00000009,CAS0000009,TXN0000010892,TXN0000010892,2026-01-03T12:22:36.000Z,2026-01-03T17:20:05.000Z,null,OPEN,MEDIUM,PENDING,QUEUE_C,SYN-R03_CROSS_BORDER,fraud_cases.csv,2026-07-31T10:05:13.306Z,d077a504-94b4-4cc6-90c3-f2e1716f872a
CASR00000010,CAS0000010,TXN-ORPHAN-000009,TXN0000198885,2026-02-01T11:03:39.000Z,2026-02-01T14:39:01.000Z,2026-02-01T23:11:32.000Z,CLOSED_LEGITIMATE,MEDIUM,LEGITIMATE,QUEUE_D,SYN-R01_NEW_DEVICE|SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,fraud_cases.csv,2026-07-31T10:05:13.306Z,1b629c98-8784-4e3e-b47f-7f21b51cfddc


## 4. Merchants — bronze_merchants → silver_merchants

In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_merchants
USING DELTA AS
WITH ranked AS (
    SELECT
        TRIM(merchant_record_id) AS merchant_record_id,
        TRIM(merchant_id) AS merchant_id,
        TRIM(merchant_label) AS merchant_label,
        TRIM(merchant_category) AS merchant_category,
        UPPER(TRIM(merchant_country_code)) AS merchant_country_code,
        UPPER(TRIM(merchant_risk_tier)) AS merchant_risk_tier,
        CAST(onboarding_date AS DATE) AS onboarding_date,
        UPPER(TRIM(merchant_status)) AS merchant_status,
        UPPER(TRIM(settlement_currency)) AS settlement_currency,
        source_file_name,
        ingestion_timestamp,
        ingestion_run_id,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_id
            ORDER BY onboarding_date DESC, ingestion_timestamp DESC
        ) AS rn
    FROM data_engineering.default.bronze_merchants
    WHERE merchant_id IS NOT NULL
      AND TRIM(merchant_id) <> ''
)
SELECT
    merchant_record_id,
    merchant_id,
    merchant_label,
    merchant_category,
    merchant_country_code,
    merchant_risk_tier,
    onboarding_date,
    merchant_status,
    settlement_currency,
    source_file_name,
    ingestion_timestamp,
    ingestion_run_id
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_merchants_count
FROM data_engineering.default.silver_merchants;


silver_merchants_count
2800


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_merchants
LIMIT 10;


merchant_record_id,merchant_id,merchant_label,merchant_category,merchant_country_code,merchant_risk_tier,onboarding_date,merchant_status,settlement_currency,source_file_name,ingestion_timestamp,ingestion_run_id
MERR0000001,MER000001,SYNTH_MERCHANT_00001,GROCERY,SG,LOW,2023-03-12,ACTIVE,SGD,merchants.csv,2026-07-31T10:05:42.587Z,4bedc7ea-1afd-4303-a039-07b059b18806
MERR0000002,MER000002,SYNTH_MERCHANT_00002,GROCERY,IN,LOW,2023-10-06,ACTIVE,INR,merchants.csv,2026-07-31T10:05:42.587Z,f04e5815-aee4-4eca-b73b-70cc6c558a78
MERR0000003,MER000003,SYNTH_MERCHANT_00003,DINING,IN,MEDIUM,2025-04-03,ACTIVE,INR,merchants.csv,2026-07-31T10:05:42.587Z,83948a17-347f-40e4-ae3f-ead95bf2db8f
MERR0000004,MER000004,SYNTH_MERCHANT_00004,GROCERY,IN,MEDIUM,2023-11-25,ACTIVE,INR,merchants.csv,2026-07-31T10:05:42.587Z,1ab37edd-edfd-4b18-8009-17cf5d0faf31
MERR0000005,MER000005,SYNTH_MERCHANT_00005,UTILITIES,IN,LOW,2022-09-03,ACTIVE,INR,merchants.csv,2026-07-31T10:05:42.587Z,61e478aa-b82e-4042-bea4-5d1d14015200
MERR0000006,MER000006,SYNTH_MERCHANT_00006,ELECTRONICS,SG,LOW,2025-01-19,ACTIVE,SGD,merchants.csv,2026-07-31T10:05:42.587Z,998dd8fb-8e7c-49df-9440-6202595a9d73
MERR0000007,MER000007,SYNTH_MERCHANT_00007,FUEL,AE,MEDIUM,2024-09-14,ACTIVE,AED,merchants.csv,2026-07-31T10:05:42.587Z,70f547d8-6eb5-46a5-9ad8-a2b86caf63be
MERR0000008,MER000008,SYNTH_MERCHANT_00008,ELECTRONICS,IN,LOW,2024-05-16,ACTIVE,INR,merchants.csv,2026-07-31T10:05:42.587Z,37e2bcbf-f20d-4a74-9060-481b2d0c5bb3
MERR0000009,MER000009,SYNTH_MERCHANT_00009,EDUCATION,GB,HIGH,2022-02-24,ACTIVE,GBP,merchants.csv,2026-07-31T10:05:42.587Z,243d8932-1c0e-4109-9834-9cce4ba7f9fd
MERR0000010,MER000010,SYNTH_MERCHANT_00010,HEALTHCARE,SG,MEDIUM,2024-07-25,ACTIVE,SGD,merchants.csv,2026-07-31T10:05:42.587Z,28a855ff-a996-4b36-b190-8de80d823c32


## 5. Transactions — bronze_transactions → silver_transactions

In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.silver_transactions
USING DELTA AS
WITH cleaned AS (
    SELECT
        TRIM(physical_record_id) AS physical_record_id,
        TRIM(transaction_id) AS transaction_id,
        TRIM(customer_id) AS customer_id,
        TRIM(account_id) AS account_id,
        TRIM(merchant_id) AS merchant_id,
        TRIM(device_id) AS device_id,
        CAST(event_timestamp AS TIMESTAMP) AS event_timestamp,
        CAST(authorization_timestamp AS TIMESTAMP) AS authorization_timestamp,
        UPPER(TRIM(transaction_status)) AS transaction_status,
        CAST(amount_original AS DECIMAL(18,2)) AS amount_original,
        UPPER(TRIM(currency)) AS currency,
        CAST(fx_rate_to_inr AS DECIMAL(18,6)) AS fx_rate_to_inr,
        CAST(amount_reporting_inr AS DECIMAL(18,2)) AS amount_reporting_inr,
        UPPER(TRIM(reporting_currency)) AS reporting_currency,
        UPPER(TRIM(channel)) AS channel,
        UPPER(TRIM(country_code)) AS country_code,
        TRIM(merchant_category) AS merchant_category,
        CAST(risk_score AS INT) AS risk_score,
        UPPER(TRIM(risk_band)) AS risk_band,
        TRIM(triggered_rule_ids) AS triggered_rule_ids,
        CAST(is_new_device AS BOOLEAN) AS is_new_device,
        CAST(location_change_flag AS BOOLEAN) AS location_change_flag,
        CAST(velocity_10m_count AS INT) AS velocity_10m_count,
        UPPER(TRIM(settlement_status)) AS settlement_status,
        CAST(settled_timestamp AS TIMESTAMP) AS settled_timestamp,
        TRIM(case_id) AS case_id,
        UPPER(TRIM(final_outcome)) AS final_outcome,
        CAST(outcome_timestamp AS TIMESTAMP) AS outcome_timestamp,
        CAST(record_updated_timestamp AS TIMESTAMP) AS record_updated_timestamp,
        source_file_name,
        ingestion_timestamp,
        ingestion_run_id
    FROM data_engineering.default.bronze_transactions
    WHERE transaction_id IS NOT NULL
      AND TRIM(transaction_id) <> ''
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY transaction_id
            ORDER BY record_updated_timestamp DESC NULLS LAST,
                     ingestion_timestamp DESC
        ) AS rn
    FROM cleaned
)
SELECT
    physical_record_id,
    transaction_id,
    customer_id,
    account_id,
    merchant_id,
    device_id,
    event_timestamp,
    authorization_timestamp,
    transaction_status,
    amount_original,
    currency,
    fx_rate_to_inr,
    amount_reporting_inr,
    reporting_currency,
    channel,
    country_code,
    merchant_category,
    risk_score,
    risk_band,
    triggered_rule_ids,
    is_new_device,
    location_change_flag,
    velocity_10m_count,
    settlement_status,
    settled_timestamp,
    case_id,
    final_outcome,
    outcome_timestamp,
    record_updated_timestamp,
    source_file_name,
    ingestion_timestamp,
    ingestion_run_id
FROM ranked
WHERE rn = 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_transactions_count
FROM data_engineering.default.silver_transactions;


silver_transactions_count
279720


In [0]:
%sql
SELECT * FROM data_engineering.default.silver_transactions
LIMIT 10;


physical_record_id,transaction_id,customer_id,account_id,merchant_id,device_id,event_timestamp,authorization_timestamp,transaction_status,amount_original,currency,fx_rate_to_inr,amount_reporting_inr,reporting_currency,channel,country_code,merchant_category,risk_score,risk_band,triggered_rule_ids,is_new_device,location_change_flag,velocity_10m_count,settlement_status,settled_timestamp,case_id,final_outcome,outcome_timestamp,record_updated_timestamp,source_file_name,ingestion_timestamp,ingestion_run_id
TXR000000001,TXN0000000001,CUS0008657,ACC00018126,MER000061,DEV00007309,2026-04-08T04:16:28.000Z,2026-04-08T04:17:49.000Z,APPROVED,17208.36,INR,1.000000,17208.36,INR,MOBILE_APP,IN,EDUCATION,29,LOW,SYN-R01_NEW_DEVICE,true,false,2,REVERSED,null,null,PENDING,null,2026-04-08T04:17:49.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,65efad6c-be46-467a-89d3-f63a9d24e59c
TXR000000002,TXN0000000002,CUS0013979,ACC00001260,MER002417,DEV00024569,2026-02-07T20:12:27.000Z,2026-02-07T20:13:37.000Z,APPROVED,127.55,AED,22.600000,2882.63,INR,MOBILE_APP,AE,EDUCATION,34,MEDIUM,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-02-07T20:51:28.000Z,null,PENDING,null,2026-02-07T20:13:37.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,73e35df8-311c-4f8d-badf-a5bc4d9529b5
TXR000000003,TXN0000000003,CUS0000656,ACC00021895,MER000175,DEV00008999,2026-06-09T10:00:31.000Z,2026-06-09T10:01:52.000Z,APPROVED,2987.08,INR,1.000000,2987.08,INR,CONTACTLESS,IN,DINING,17,LOW,SYN-R00_BASELINE,false,false,3,SETTLED,2026-06-10T16:55:53.000Z,null,PENDING,null,2026-06-09T10:01:52.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,3f2576ea-6248-4458-b5ab-2eae9841e09d
TXR000000004,TXN0000000004,CUS0010706,ACC00006951,MER000068,DEV00000516,2026-05-05T15:33:42.000Z,2026-05-05T15:33:43.000Z,APPROVED,232.81,USD,83.000000,19323.23,INR,CONTACTLESS,US,ENTERTAINMENT,20,LOW,SYN-R03_CROSS_BORDER,false,false,1,SETTLED,2026-05-06T16:47:08.000Z,null,PENDING,null,2026-05-05T15:33:43.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,fb39480d-8b27-4ab5-a662-f604aef8a7b7
TXR000000005,TXN0000000005,CUS0013471,ACC00011798,MER001607,DEV00009221,2026-06-02T20:10:23.000Z,2026-06-02T20:11:33.000Z,APPROVED,17.22,GBP,105.000000,1808.10,INR,POS,GB,FUEL,20,LOW,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-06-04T14:56:30.000Z,null,PENDING,null,2026-06-02T20:11:33.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,55461cc6-0236-483b-9434-cc84405ef4f3
TXR000000006,TXN0000000006,CUS0000618,ACC00015448,MER001685,DEV00025323,2026-04-21T11:25:32.000Z,2026-04-21T11:27:27.000Z,APPROVED,79.27,SGD,62.000000,4914.74,INR,CONTACTLESS,SG,MARKETPLACE,36,MEDIUM,SYN-R03_CROSS_BORDER|SYN-R06_LOCATION_CHANGE,false,true,2,SETTLED,2026-04-24T04:52:19.000Z,null,PENDING,null,2026-04-21T11:27:27.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,4f369a5e-abc1-4989-983b-812ab08a9be4
TXR000000007,TXN0000000007,CUS0010604,ACC00008545,MER002099,DEV00019472,2026-05-25T20:58:19.000Z,2026-05-25T20:58:22.000Z,APPROVED,15207.41,INR,1.000000,15207.41,INR,ECOMMERCE,IN,ELECTRONICS,3,LOW,SYN-R00_BASELINE,false,false,1,SETTLED,2026-05-27T02:22:28.000Z,null,PENDING,null,2026-05-25T20:58:22.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,29375fcd-7ae9-4931-8a97-fb314e144f78
TXR000000008,TXN0000000008,CUS0013366,ACC00011839,MER000310,DEV00010273,2026-03-05T08:42:33.000Z,2026-03-05T08:43:15.000Z,APPROVED,579.24,USD,83.000000,48076.92,INR,ECOMMERCE,US,TRAVEL,33,MEDIUM,SYN-R04_MERCHANT_RISK,false,false,3,SETTLED,2026-03-06T06:01:13.000Z,null,PENDING,null,2026-03-05T08:43:15.000Z,transactions_databricks_compatible (1).parquet,2026-07-31T10:06:08.228Z,19dbc62a-3fea-4200-ad57-4ad15cdab0fd
TXR000000009,TXN0000000009,CUS0005552,ACC00014082,MER000880,DEV00024623,2026-04-10T20:26:05.000Z,2026-04-10T20:26:48.000Z,APPROVED

## 6. Silver Layer Validation

In [0]:
%sql
SELECT
    'silver_customers' AS table_name,
    COUNT(*) AS row_count
FROM data_engineering.default.silver_customers

UNION ALL

SELECT
    'silver_customer_accounts',
    COUNT(*)
FROM data_engineering.default.silver_customer_accounts

UNION ALL

SELECT
    'silver_devices',
    COUNT(*)
FROM data_engineering.default.silver_devices

UNION ALL

SELECT
    'silver_fraud_cases',
    COUNT(*)
FROM data_engineering.default.silver_fraud_cases

UNION ALL

SELECT
    'silver_merchants',
    COUNT(*)
FROM data_engineering.default.silver_merchants

UNION ALL

SELECT
    'silver_transactions',
    COUNT(*)
FROM data_engineering.default.silver_transactions;


table_name,row_count
silver_customers,18000
silver_customer_accounts,22000
silver_devices,30000
silver_fraud_cases,4200
silver_merchants,2800
silver_transactions,279720


In [0]:
%sql
SELECT
    COUNT(*) AS null_transaction_ids
FROM data_engineering.default.silver_transactions
WHERE transaction_id IS NULL;


null_transaction_ids
0


In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_transaction_ids
FROM (
    SELECT transaction_id
    FROM data_engineering.default.silver_transactions
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
);


duplicate_transaction_ids
0


In [0]:
%sql
SELECT
    COUNT(*) AS null_customer_ids
FROM data_engineering.default.silver_customers
WHERE customer_id IS NULL;


null_customer_ids
0


In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_customer_ids
FROM (
    SELECT customer_id
    FROM data_engineering.default.silver_customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
);


duplicate_customer_ids
0


## Silver Tables Created

- `data_engineering.default.silver_customers`
- `data_engineering.default.silver_customer_accounts`
- `data_engineering.default.silver_devices`
- `data_engineering.default.silver_fraud_cases`
- `data_engineering.default.silver_merchants`
- `data_engineering.default.silver_transactions`

The Bronze source tables and their names are taken directly from the uploaded Bronze notebook. fileciteturn2file0L99-L108 fileciteturn2file0L195-L205 fileciteturn2file0L293-L302 fileciteturn2file0L390-L399

The Bronze notebook creates the source tables with Delta and adds `source_file_name`, `ingestion_timestamp`, and `ingestion_run_id`, which this Silver notebook carries forward as lineage columns. fileciteturn2file2L567-L578 fileciteturn2file2L579-L590 fileciteturn2file2L591-L602
